## Regression with Intermediate Neural Network 
Build a intermediate-depth neural network for regression task

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch 
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

In [6]:
# Load the Boston housing dataset and create train/test split
X, y = fetch_openml(name="boston", version=1, as_frame=False, return_X_y=True)
y = y.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

# Standardize features so the regression model trains more reliably
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Number of samples model processes at once during training/testing 
batch_size = 64 

# Create data loaders to handle batching and shuffling of data during training/testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 404
Test size: 102


In [13]:
model = nn.Sequential(            
    nn.Linear(13, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(16, 1) # Output layer for regression (single continuous value)
)

In [14]:
print(model)
total_params = sum(p.numel() for p in model.parameters())
print("Total model parameters:", total_params)

Sequential(
  (0): Linear(in_features=13, out_features=32, bias=True)
  (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Linear(in_features=32, out_features=16, bias=True)
  (4): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU()
  (6): Dropout(p=0.2, inplace=False)
  (7): Linear(in_features=16, out_features=1, bias=True)
)
Total model parameters: 1089


In [15]:
criterion = nn.MSELoss()  # Mean Squared Error loss for regression
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [17]:
# Model training loop with GPU support if available
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
model = model.to(device)

epochs = 30
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    total = 0

    # Iterate over mini-batches from the training loader
    for features, targets in train_loader:
        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)
        total += targets.size(0)

    train_loss = running_loss / total

    # Evaluate on the test split without computing gradients
    model.eval()
    test_running_loss = 0.0
    test_total = 0
    with torch.no_grad():
        for features, targets in test_loader:
            features = features.to(device)
            targets = targets.to(device)
            outputs = model(features)
            loss = criterion(outputs, targets)
            test_running_loss += loss.item() * features.size(0)
            test_total += targets.size(0)

    test_loss = test_running_loss / test_total
    print(f"Epoch {epoch + 1}/{epochs} - train loss: {train_loss:.4f} - test loss: {test_loss:.4f}")

Using device: mps
Epoch 1/30 - train loss: 19.3678 - test loss: 12.4850
Epoch 2/30 - train loss: 25.6331 - test loss: 12.0957
Epoch 3/30 - train loss: 26.0157 - test loss: 11.4140
Epoch 4/30 - train loss: 24.2099 - test loss: 13.8181
Epoch 5/30 - train loss: 25.7427 - test loss: 13.5074
Epoch 6/30 - train loss: 25.8644 - test loss: 11.8614
Epoch 7/30 - train loss: 24.9568 - test loss: 13.6422
Epoch 8/30 - train loss: 23.1812 - test loss: 16.2689
Epoch 9/30 - train loss: 21.0805 - test loss: 19.8240
Epoch 10/30 - train loss: 19.6728 - test loss: 14.1969
Epoch 11/30 - train loss: 23.3756 - test loss: 14.1659
Epoch 12/30 - train loss: 24.9473 - test loss: 11.8704
Epoch 13/30 - train loss: 22.9109 - test loss: 10.2429
Epoch 14/30 - train loss: 22.7144 - test loss: 13.2807
Epoch 15/30 - train loss: 25.5822 - test loss: 13.0309
Epoch 16/30 - train loss: 21.6641 - test loss: 11.1631
Epoch 17/30 - train loss: 20.8947 - test loss: 13.9193
Epoch 18/30 - train loss: 18.5473 - test loss: 11.0458
E

In [ ]:
model.eval()
with torch.no_grad():
    sample_predictions = model(X_test[:5].to(device))

print(sample_predictions.cpu())

AttributeError: 'Sequential' object has no attribute 'predict'